# IC50 Baseline Models Training - Hyperparameter Grid Search

**Testing:**
- 9 hyperparameter combinations (3 learning rates × 3 dropout rates) for each model
- 2 model types (MLP vs GNN)
- 2 split strategies (Random vs Scaffold)
- **Total: 36 experiments**

**Models:**
- MLP: Feedforward network on Morgan fingerprints (classical approach)
- GNN: Graph Convolutional Network on molecular graphs (modern approach)

**Evaluation:**
- Random split (80/10/10) - tests performance on similar distribution
- Scaffold split (80/10/10) - tests generalization to new chemical scaffolds

## Section 1: Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, BatchNorm, global_mean_pool
from torch_geometric.loader import DataLoader as PyGDataLoader

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
def seed_everything(seed: int = 42):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Make PyTorch operations deterministic (slower but reproducible)
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# Device configuration - automatically uses GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Display configuration for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Section 2: Data File Configuration

**Configure your data files here!**

Specify the exact filenames for:
- Source data (single parquet file)
- MLP features (can be multiple part files)
- GNN graphs (can be multiple part files)

In [ ]:
def load_source_data(data_dir: str, source_file: str) -> pd.DataFrame:
    file_path = Path(data_dir) / source_file
    
    if not file_path.exists():
        raise FileNotFoundError(f"Source file not found: {file_path}")
    
    df = pd.read_parquet(file_path)
    df = df.dropna(subset=['canonical_smiles', 'pic50'])
    
    print(f"Loaded {len(df)} samples from {source_file}")
    print(f"Columns: {df.columns.tolist()}")
    
    return df


def load_mlp_features(data_dir: str, feature_files: List[str]) -> Dict[int, np.ndarray]:
    data_path = Path(data_dir)
    
    # Check all files exist
    for fname in feature_files:
        fpath = data_path / fname
        if not fpath.exists():
            raise FileNotFoundError(f"MLP feature file not found: {fpath}")
    
    # Load and concatenate all files
    dfs = []
    for fname in feature_files:
        fpath = data_path / fname
        print(f"  Loading {fname}...")
        dfs.append(pd.read_parquet(fpath))
    
    df = pd.concat(dfs, ignore_index=True)
    
    # Create dictionary mapping activity_id to fingerprint
    features_dict = {}
    for _, row in df.iterrows():
        features_dict[row['activity_id']] = np.array(row['fingerprint'], dtype=np.float32)
    
    print(f"Loaded MLP features for {len(features_dict)} samples from {len(feature_files)} file(s)")
    sample_fp = next(iter(features_dict.values()))
    print(f"Fingerprint dimension: {len(sample_fp)}")
    
    return features_dict


def load_gnn_graphs(data_dir: str, graph_files: List[str]) -> Dict[int, Data]:
    data_path = Path(data_dir)
    
    # Check all files exist
    for fname in graph_files:
        fpath = data_path / fname
        if not fpath.exists():
            raise FileNotFoundError(f"GNN graph file not found: {fpath}")
    
    # Load all graph lists and concatenate
    all_graphs = []
    for fname in graph_files:
        fpath = data_path / fname
        print(f"  Loading {fname}...")
        graphs = torch.load(fpath, weights_only=False)
        all_graphs.extend(graphs)
    
    # Create dictionary mapping activity_id to graph
    graphs_dict = {}
    for graph in all_graphs:
        if hasattr(graph, 'activity_id'):
            graphs_dict[int(graph.activity_id)] = graph
    
    print(f"Loaded GNN graphs for {len(graphs_dict)} samples from {len(graph_files)} file(s)")
    sample_graph = next(iter(graphs_dict.values()))
    print(f"Sample graph - nodes: {sample_graph.num_nodes}, edges: {sample_graph.num_edges}")
    print(f"Node features: {sample_graph.x.shape}, Edge features: {sample_graph.edge_attr.shape if sample_graph.edge_attr is not None else 'None'}")
    
    return graphs_dict


def join_datasets(source_df: pd.DataFrame, 
                  mlp_features: Dict[int, np.ndarray],
                  gnn_graphs: Dict[int, Data]) -> Tuple[pd.DataFrame, Dict, Dict]:
    # Join all datasets on common activity_ids
    
    # Find common activity_ids across all datasets
    source_ids = set(source_df['activity_id'].values)
    mlp_ids = set(mlp_features.keys())
    gnn_ids = set(gnn_graphs.keys())
    
    common_ids = source_ids & mlp_ids & gnn_ids
    
    print(f"\nDataset intersection:")
    print(f"Source data: {len(source_ids)} samples")
    print(f"MLP features: {len(mlp_ids)} samples")
    print(f"GNN graphs: {len(gnn_ids)} samples")
    print(f"Common samples: {len(common_ids)}")
    
    # Filter to common IDs
    filtered_df = source_df[source_df['activity_id'].isin(common_ids)].copy()
    filtered_mlp = {aid: mlp_features[aid] for aid in common_ids}
    filtered_gnn = {aid: gnn_graphs[aid] for aid in common_ids}
    
    # Validation
    assert len(filtered_df) == len(common_ids), "DataFrame filtering failed"
    assert len(filtered_mlp) == len(common_ids), "MLP features filtering failed"
    assert len(filtered_gnn) == len(common_ids), "GNN graphs filtering failed"
    
    print(f"\nFinal dataset: {len(filtered_df)} samples")
    print(f"pIC50 range: [{filtered_df['pic50'].min():.2f}, {filtered_df['pic50'].max():.2f}]")
    print(f"pIC50 mean: {filtered_df['pic50'].mean():.2f} ± {filtered_df['pic50'].std():.2f}")
    
    return filtered_df, filtered_mlp, filtered_gnn

In [ ]:
# Load all data using configured files
print("Loading source data...")
source_df = load_source_data(DATA_DIR, SOURCE_FILE)

print("\nLoading MLP features...")
mlp_features = load_mlp_features(DATA_DIR, MLP_FEATURE_FILES)

print("\nLoading GNN graphs...")
gnn_graphs = load_gnn_graphs(DATA_DIR, GNN_GRAPH_FILES)

# Join datasets
final_df, mlp_features_dict, gnn_graphs_dict = join_datasets(source_df, mlp_features, gnn_graphs)

print("\nData loading complete!")
print(final_df.head())

## Section 4: Configuration Grid

We test different hyperparameter combinations to find the best configuration.

**Why these parameters matter:**
- **Learning rate** - controls how fast the model learns. Too high = unstable, too low = slow/stuck
- **Dropout** - randomly turns off neurons during training to prevent overfitting (memorizing training data)

**Total experiments:** 9 configs × 2 models × 2 splits = 36 experiments

In [ ]:
# Load all data
source_df = load_source_data()
mlp_features = load_mlp_features()
gnn_graphs = load_gnn_graphs()

# Join datasets
final_df, mlp_features_dict, gnn_graphs_dict = join_datasets(source_df, mlp_features, gnn_graphs)

print("\nData loading complete!")
print(final_df.head())

## Section 5: Data Splitting

We split data into train/validation/test sets (80/10/10).

**Two splitting strategies:**
1. **Random split** - shuffle all molecules randomly
   - Easier task: train and test have similar molecules
   - Tests: "can the model learn from examples?"

2. **Scaffold split** - group by chemical core structure
   - Harder task: test molecules have different core structures than training
   - Tests: "can the model generalize to truly new molecules?"
   - More realistic for drug discovery!

In [ ]:
# Hyperparameter grid
learning_rates = [1e-3, 3e-4, 1e-4]  # 0.001, 0.0003, 0.0001
dropout_rates = [0.1, 0.2, 0.3]       # 10%, 20%, 30%
batch_size = 128                       # Fixed - number of molecules processed at once

# Fixed hyperparameters (same for all experiments)
weight_decay = 1e-5        # L2 regularization strength
patience = 10              # Early stopping patience (stop if no improvement for N epochs)
max_epochs = 50            # Maximum training epochs
gradient_clip = 1.0        # Maximum gradient norm (prevents exploding gradients)

# Generate all configurations
configs = []
for lr in learning_rates:
    for dropout in dropout_rates:
        configs.append({
            'lr': lr,
            'dropout': dropout,
            'batch_size': batch_size,
            'weight_decay': weight_decay,
            'patience': patience,
            'max_epochs': max_epochs
        })

print("Configuration Grid Summary")
print("=" * 50)
print(f"Learning rates: {len(learning_rates)} values - {learning_rates}")
print(f"Dropout rates: {len(dropout_rates)} values - {dropout_rates}")
print(f"Batch size: {batch_size} (fixed)")
print(f"Configurations per model: {len(configs)}")
print(f"Total experiments: {len(configs) * 4} (9 × 4 model/split combos)")
print(f"\nEstimated time (GPU): 2-4 hours")
print(f"Estimated time (CPU): 15-20 hours (NOT recommended!)")
print("=" * 50)

## Section 4: Data Splitting

We split data into train/validation/test sets (80/10/10).

**Two splitting strategies:**
1. **Random split** - shuffle all molecules randomly
   - Easier task: train and test have similar molecules
   - Tests: "can the model learn from examples?"

2. **Scaffold split** - group by chemical core structure
   - Harder task: test molecules have different core structures than training
   - Tests: "can the model generalize to truly new molecules?"
   - More realistic for drug discovery!

In [ ]:
def random_split(df: pd.DataFrame, 
                 test_size: float = 0.1, 
                 val_size: float = 0.1, 
                 seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Randomly split data into train/val/test sets."""
    # Shuffle to prevent any ordering bias
    df_shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    n = len(df_shuffled)
    n_test = int(n * test_size)
    n_val = int(n * val_size)
    n_train = n - n_test - n_val
    
    train_df = df_shuffled[:n_train]
    val_df = df_shuffled[n_train:n_train + n_val]
    test_df = df_shuffled[n_train + n_val:]
    
    print(f"Random split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
    
    return train_df, val_df, test_df


def scaffold_split(df: pd.DataFrame,
                   test_size: float = 0.1,
                   val_size: float = 0.1,
                   seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split data by Murcko scaffolds to test generalization.
    
    Groups molecules by their core structure (scaffold), then allocates entire
    scaffolds to different splits. This ensures train/val/test have different
    chemical scaffolds, testing true generalization ability.
    """
    # Compute Murcko scaffold for each molecule
    scaffolds = defaultdict(list)
    
    for idx, row in df.iterrows():
        smiles = row['canonical_smiles']
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
                scaffolds[scaffold].append(idx)
        except:
            # If scaffold computation fails, use SMILES as scaffold
            scaffolds[smiles].append(idx)
    
    # Sort scaffolds by size (descending) for more balanced splits
    scaffold_sets = [(scaffold, set(indices)) for scaffold, indices in scaffolds.items()]
    scaffold_sets.sort(key=lambda x: len(x[1]), reverse=True)
    
    # Greedy allocation to reach target split sizes
    n_total = len(df)
    n_test_target = int(n_total * test_size)
    n_val_target = int(n_total * val_size)
    
    train_indices = []
    val_indices = []
    test_indices = []
    
    for scaffold, indices in scaffold_sets:
        if len(test_indices) < n_test_target:
            test_indices.extend(indices)
        elif len(val_indices) < n_val_target:
            val_indices.extend(indices)
        else:
            train_indices.extend(indices)
    
    # Create split DataFrames
    train_df = df.loc[train_indices].reset_index(drop=True)
    val_df = df.loc[val_indices].reset_index(drop=True)
    test_df = df.loc[test_indices].reset_index(drop=True)
    
    print(f"Scaffold split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
    print(f"Number of unique scaffolds: {len(scaffolds)}")
    
    return train_df, val_df, test_df

## Section 6: Model Definitions

We define two model architectures:

### MLP (Multi-Layer Perceptron)
Simple feedforward network that processes molecular fingerprints.
- Input: 2048-dimensional fingerprint (bit vector)
- Architecture: 2048 → 512 → 128 → 1
- Activation: ReLU (turns negative values to zero)
- Regularization: Dropout (randomly turns off neurons during training)

### GNN (Graph Neural Network)
Advanced network that processes molecular graphs.
- Input: molecular graph (atoms=nodes, bonds=edges)
- Architecture: Node encoding → 3× GCN layers → Pooling → Prediction head
- Key advantage: understands molecular structure directly

In [ ]:
# Create both split types
print("Creating random split...")
train_random, val_random, test_random = random_split(final_df, seed=42)

print("\nCreating scaffold split...")
train_scaffold, val_scaffold, test_scaffold = scaffold_split(final_df, seed=42)

## Section 5: Model Definitions

We define two model architectures:

### MLP (Multi-Layer Perceptron)
Simple feedforward network that processes molecular fingerprints.
- Input: 2048-dimensional fingerprint (bit vector)
- Architecture: 2048 → 512 → 128 → 1
- Activation: ReLU (turns negative values to zero)
- Regularization: Dropout (randomly turns off neurons during training)

### GNN (Graph Neural Network)
Advanced network that processes molecular graphs.
- Input: molecular graph (atoms=nodes, bonds=edges)
- Architecture: Node encoding → 3× GCN layers → Pooling → Prediction head
- Key advantage: understands molecular structure directly

## Section 7: Training Functions

This is where the magic happens! The training loop:
1. Shows the model batches of molecules
2. Model makes predictions
3. Calculate error (how wrong was it?)
4. Update model weights to do better
5. Repeat until model stops improving

**Early stopping:** If validation performance doesn't improve for 10 epochs, we stop.
This prevents overfitting (memorizing training data instead of learning patterns).

In [ ]:
class GNNBaseline(nn.Module):
    """GNN baseline for pIC50 prediction from molecular graphs.
    
    Uses Graph Convolutional Networks (GCN) to learn from molecular structure.
    Key components:
    - GCN layers: propagate information between connected atoms
    - Batch normalization: stabilizes training
    - Residual connections: helps gradient flow in deep networks
    - Global pooling: aggregates atom features to molecule-level representation
    """
    
    def __init__(self, 
                 node_feature_dim: int,
                 edge_feature_dim: int,
                 hidden_dim: int = 64,
                 num_layers: int = 3,
                 dropout: float = 0.10,
                 pooling: str = 'mean'):
        super().__init__()
        
        if pooling not in {'mean', 'add'}:
            raise ValueError("pooling must be 'mean' or 'add'")
        
        self.pooling = pooling
        self.dropout = dropout
        
        # Project node features to hidden dimension
        self.node_proj = nn.Linear(node_feature_dim, hidden_dim)
        
        # Edge feature encoder (optional, but helps with bond information)
        self.edge_encoder = nn.Sequential(
            nn.Linear(edge_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        
        # GCN layers with batch normalization
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.norms.append(BatchNorm(hidden_dim))
        
        # Prediction head - converts molecule representation to pIC50
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
    
    def forward(self, data: Batch) -> torch.Tensor:
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # Encode node features
        x = self.node_proj(x)
        
        # Apply GCN layers with residual connections
        # Residual: x_new = f(x) + x helps gradient flow
        for conv, norm in zip(self.convs, self.norms):
            residual = x
            x = conv(x, edge_index)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + residual  # Residual connection
        
        # Global pooling: aggregate all atoms in each molecule
        # This converts node-level features to graph-level features
        x = global_mean_pool(x, batch)
        
        # Prediction head
        return self.head(x).squeeze(-1)

## Section 8: Experiments - Grid Search

Now we run all 36 experiments! For each combination of:
- Model type (MLP or GNN)
- Split type (Random or Scaffold)
- Configuration (9 hyperparameter combos)

We train a model from scratch and record the results.

This will take 2-4 hours on GPU or 15-20 hours on CPU.

In [ ]:
def train_one_epoch(model: nn.Module,
                    loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    criterion: nn.Module,
                    device: torch.device,
                    scaler,
                    is_gnn: bool = False) -> float:
    """Train model for one epoch with mixed precision."""
    model.train()  # Enable dropout
    total_loss = 0
    
    for batch in loader:
        if is_gnn:
            batch = batch.to(device)
            targets = batch.y
        else:
            features, targets = batch
            features = features.to(device)
            targets = targets.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision: use float16 for forward pass (faster, less memory)
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            predictions = model(batch if is_gnn else features)
            loss = criterion(predictions, targets)
        
        # Backward pass with gradient scaling (handles float16 gradients)
        scaler.scale(loss).backward()
        
        # Gradient clipping: prevent exploding gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item() * len(targets)
    
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss


@torch.no_grad()
def evaluate(model: nn.Module,
             loader: DataLoader,
             criterion: nn.Module,
             device: torch.device,
             is_gnn: bool = False) -> Tuple[float, float]:
    """Evaluate model on validation/test set."""
    model.eval()  # Disable dropout
    total_loss = 0
    all_predictions = []
    all_targets = []
    
    for batch in loader:
        if is_gnn:
            batch = batch.to(device)
            predictions = model(batch)
            targets = batch.y
        else:
            features, targets = batch
            features = features.to(device)
            targets = targets.to(device)
            predictions = model(features)
        
        loss = criterion(predictions, targets)
        total_loss += loss.item() * len(targets)
        
        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    r2 = r2_score(all_targets, all_predictions)
    
    return avg_loss, r2


def train_and_score(model: nn.Module,
                    train_loader: DataLoader,
                    val_loader: DataLoader,
                    test_loader: DataLoader,
                    config: Dict,
                    device: torch.device = device,
                    is_gnn: bool = False,
                    print_every: int = 5) -> Dict:
    """Train model with early stopping and return results."""
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), 
                                 lr=config['lr'], 
                                 weight_decay=config['weight_decay'])
    
    # Gradient scaler for mixed precision training
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
    
    best_val_loss = float('inf')
    best_epoch = 0
    epochs_without_improvement = 0
    best_model_state = None
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_r2': []
    }
    
    for epoch in range(config['max_epochs']):
        # Training
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, 
                                    device, scaler, is_gnn)
        
        # Validation
        val_loss, val_r2 = evaluate(model, val_loader, criterion, device, is_gnn)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_r2'].append(val_r2)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            best_model_state = model.state_dict().copy()
        else:
            epochs_without_improvement += 1
        
        # Print progress
        if (epoch + 1) % print_every == 0:
            mem_str = ""
            if device.type == 'cuda':
                mem_allocated = torch.cuda.memory_allocated() / 1e9
                mem_str = f" | GPU: {mem_allocated:.1f}GB"
            print(f"  Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val R²: {val_r2:.4f}{mem_str}")
        
        # Early stopping
        if epochs_without_improvement >= config['patience']:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    # Restore best model
    model.load_state_dict(best_model_state)
    
    # Final evaluation on validation and test sets
    val_loss, val_r2 = evaluate(model, val_loader, criterion, device, is_gnn)
    test_loss, test_r2 = evaluate(model, test_loader, criterion, device, is_gnn)
    
    return {
        'model': model,
        'best_epoch': best_epoch + 1,
        'total_epochs': epoch + 1,
        'val_loss': val_loss,
        'val_r2': val_r2,
        'test_loss': test_loss,
        'test_r2': test_r2,
        'history': history,
        'config': config
    }

## Section 7: Experiments - Grid Search

Now we run all 36 experiments! For each combination of:
- Model type (MLP or GNN)
- Split type (Random or Scaffold)
- Configuration (9 hyperparameter combos)

We train a model from scratch and record the results.

This will take 2-4 hours on GPU or 15-20 hours on CPU.

## Section 9: Results Analysis

Now let's analyze the results to find the best configurations!

In [ ]:
# Run all experiments
all_results = []
experiment_num = 0
total_experiments = len(configs) * 4  # 9 configs × 4 model/split combos

start_time = time.time()

for model_type in ['MLP', 'GNN']:
    for split_type in ['Random', 'Scaffold']:
        # Select appropriate data splits
        if split_type == 'Random':
            train_df, val_df, test_df = train_random, val_random, test_random
        else:
            train_df, val_df, test_df = train_scaffold, val_scaffold, test_scaffold
        
        # Create data loaders
        if model_type == 'MLP':
            train_loader, val_loader, test_loader = create_mlp_loaders(
                train_df, val_df, test_df, mlp_features_dict, batch_size=128
            )
        else:
            train_loader, val_loader, test_loader = create_gnn_loaders(
                train_df, val_df, test_df, gnn_graphs_dict, batch_size=128
            )
        
        # Test each configuration
        for config in configs:
            experiment_num += 1
            
            print(f"\n{'='*80}")
            print(f"Experiment {experiment_num}/{total_experiments}: {model_type} + {split_type} Split")
            print(f"Config: lr={config['lr']}, dropout={config['dropout']}")
            print(f"{'='*80}")
            
            # Create model
            if model_type == 'MLP':
                model = MLPBaseline(input_size=2048, dropout=config['dropout'])
            else:
                model = GNNBaseline(
                    node_feature_dim=node_feature_dim,
                    edge_feature_dim=edge_feature_dim,
                    hidden_dim=64,
                    num_layers=3,
                    dropout=config['dropout']
                )
            
            # Train and evaluate
            results = train_and_score(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                test_loader=test_loader,
                config=config,
                device=device,
                is_gnn=(model_type == 'GNN')
            )
            
            print(f"\n  Best epoch {results['best_epoch']}: "
                  f"Val R²={results['val_r2']:.4f}, Test R²={results['test_r2']:.4f}")
            
            # Store results
            all_results.append({
                'Model': model_type,
                'Split Type': split_type,
                'Learning Rate': config['lr'],
                'Dropout': config['dropout'],
                'Batch Size': config['batch_size'],
                'R² (Val)': results['val_r2'],
                'R² (Test)': results['test_r2'],
                'MSE (Val)': results['val_loss'],
                'MSE (Test)': results['test_loss'],
                'Epochs Trained': results['total_epochs'],
                'Best Epoch': results['best_epoch']
            })
            
            # Clear GPU cache between experiments to prevent OOM
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            
            # Estimate time remaining
            elapsed = time.time() - start_time
            avg_time_per_exp = elapsed / experiment_num
            remaining_exps = total_experiments - experiment_num
            estimated_remaining = avg_time_per_exp * remaining_exps
            print(f"  Estimated time remaining: {estimated_remaining/60:.1f} minutes")

total_time = time.time() - start_time
print(f"\n{'='*80}")
print(f"All experiments complete! Total time: {total_time/3600:.2f} hours")
print(f"{'='*80}")

## Section 8: Results Analysis

Now let's analyze the results to find the best configurations!

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(all_results)

# Sort by Test R² (best first)
results_df_sorted = results_df.sort_values('R² (Test)', ascending=False).reset_index(drop=True)

print("=" * 120)
print("FULL RESULTS - ALL 36 EXPERIMENTS")
print("=" * 120)
print(results_df_sorted.to_string(index=False))
print("\n")

In [ ]:
# Best configuration per model/split combination
print("=" * 120)
print("BEST CONFIGURATION PER MODEL/SPLIT TYPE")
print("=" * 120)

best_configs = []
for model in ['MLP', 'GNN']:
    for split in ['Random', 'Scaffold']:
        subset = results_df[(results_df['Model'] == model) & (results_df['Split Type'] == split)]
        best = subset.nlargest(1, 'R² (Test)')
        best_configs.append(best)

best_configs_df = pd.concat(best_configs).reset_index(drop=True)
print(best_configs_df.to_string(index=False))
print("\n")

In [ ]:
# Learning rate analysis
print("=" * 80)
print("LEARNING RATE ANALYSIS - Average Test R² for Each Learning Rate")
print("=" * 80)

lr_analysis = results_df.groupby('Learning Rate')['R² (Test)'].agg(['mean', 'std', 'min', 'max'])
lr_analysis.columns = ['Mean R²', 'Std Dev', 'Min R²', 'Max R²']
lr_analysis = lr_analysis.sort_values('Mean R²', ascending=False)
print(lr_analysis)
print("\n")

## Section 10: Visualizations

Visual analysis of the results to identify patterns.

In [ ]:
# Key takeaways
print("=" * 80)
print("KEY TAKEAWAYS")
print("=" * 80)

# 1. Best model on random split
best_random = results_df[results_df['Split Type'] == 'Random'].nlargest(1, 'R² (Test)')
print(f"\n1. Best model on random split: {best_random['Model'].values[0]}")
print(f"   - Config: lr={best_random['Learning Rate'].values[0]}, dropout={best_random['Dropout'].values[0]}")
print(f"   - Test R²: {best_random['R² (Test)'].values[0]:.4f}")

# 2. Best model on scaffold split
best_scaffold = results_df[results_df['Split Type'] == 'Scaffold'].nlargest(1, 'R² (Test)')
print(f"\n2. Best model on scaffold split: {best_scaffold['Model'].values[0]}")
print(f"   - Config: lr={best_scaffold['Learning Rate'].values[0]}, dropout={best_scaffold['Dropout'].values[0]}")
print(f"   - Test R²: {best_scaffold['R² (Test)'].values[0]:.4f}")

# 3. Generalization gap
mlp_random = results_df[(results_df['Model'] == 'MLP') & (results_df['Split Type'] == 'Random')]['R² (Test)'].max()
mlp_scaffold = results_df[(results_df['Model'] == 'MLP') & (results_df['Split Type'] == 'Scaffold')]['R² (Test)'].max()
gnn_random = results_df[(results_df['Model'] == 'GNN') & (results_df['Split Type'] == 'Random')]['R² (Test)'].max()
gnn_scaffold = results_df[(results_df['Model'] == 'GNN') & (results_df['Split Type'] == 'Scaffold')]['R² (Test)'].max()

mlp_gap = mlp_random - mlp_scaffold
gnn_gap = gnn_random - gnn_scaffold

print(f"\n3. Generalization gap (Random R² - Scaffold R²):")
print(f"   - MLP: {mlp_gap:.4f}")
print(f"   - GNN: {gnn_gap:.4f}")

# 4. GNN vs MLP comparison
gnn_advantage_random = gnn_random - mlp_random
gnn_advantage_scaffold = gnn_scaffold - mlp_scaffold

print(f"\n4. GNN advantage over MLP (GNN R² - MLP R²):")
print(f"   - Random split: {gnn_advantage_random:+.4f}")
print(f"   - Scaffold split: {gnn_advantage_scaffold:+.4f}")

if gnn_advantage_scaffold > 0 and gnn_advantage_random > 0:
    print("   → GNN consistently outperforms MLP on both splits!")
elif gnn_advantage_random > 0:
    print("   → GNN performs better on similar data, but advantage shrinks on new scaffolds")
else:
    print("   → MLP outperforms GNN in this setup")

# 5. Optimal hyperparameter ranges
best_lr = lr_analysis.index[0]
best_dropout = dropout_analysis.index[0]

print(f"\n5. Optimal hyperparameter ranges:")
print(f"   - Best learning rate: {best_lr}")
print(f"   - Best dropout: {best_dropout}")
print(f"   - Learning rate ranking: {', '.join([str(lr) for lr in lr_analysis.index])}")
print(f"   - Dropout ranking: {', '.join([str(d) for d in dropout_analysis.index])}")

print("\n" + "=" * 80)

## Section 9: Visualizations

Visual analysis of the results to identify patterns.

In [ ]:
# 1. Configuration comparison
fig, ax = plt.subplots(1, 1, figsize=(16, 6))

# Create labels for each configuration
results_df_sorted['config_label'] = (results_df_sorted['Model'] + '-' + 
                                     results_df_sorted['Split Type'] + '\n' +
                                     'lr=' + results_df_sorted['Learning Rate'].astype(str) + '\n' +
                                     'dp=' + results_df_sorted['Dropout'].astype(str))

# Color by model type
colors = ['#3498db' if m == 'MLP' else '#2ecc71' for m in results_df_sorted['Model']]

ax.bar(range(len(results_df_sorted)), results_df_sorted['R² (Test)'], color=colors, alpha=0.7)
ax.set_xlabel('Configuration', fontsize=12)
ax.set_ylabel('Test R²', fontsize=12)
ax.set_title('Test R² for All 36 Configurations (Blue=MLP, Green=GNN)', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(results_df_sorted)))
ax.set_xticklabels(results_df_sorted['config_label'], rotation=90, fontsize=7)
ax.grid(axis='y', alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3498db', alpha=0.7, label='MLP'),
                   Patch(facecolor='#2ecc71', alpha=0.7, label='GNN')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Learning rate vs performance (box plots)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MLP
mlp_data = results_df[results_df['Model'] == 'MLP']
mlp_lr_groups = [mlp_data[mlp_data['Learning Rate'] == lr]['R² (Test)'].values 
                 for lr in learning_rates]
axes[0].boxplot(mlp_lr_groups, labels=[str(lr) for lr in learning_rates])
axes[0].set_xlabel('Learning Rate', fontsize=12)
axes[0].set_ylabel('Test R²', fontsize=12)
axes[0].set_title('MLP: Learning Rate vs Performance', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# GNN
gnn_data = results_df[results_df['Model'] == 'GNN']
gnn_lr_groups = [gnn_data[gnn_data['Learning Rate'] == lr]['R² (Test)'].values 
                 for lr in learning_rates]
axes[1].boxplot(gnn_lr_groups, labels=[str(lr) for lr in learning_rates])
axes[1].set_xlabel('Learning Rate', fontsize=12)
axes[1].set_ylabel('Test R²', fontsize=12)
axes[1].set_title('GNN: Learning Rate vs Performance', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 3. Dropout vs performance (box plots)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MLP
mlp_dropout_groups = [mlp_data[mlp_data['Dropout'] == d]['R² (Test)'].values 
                      for d in dropout_rates]
axes[0].boxplot(mlp_dropout_groups, labels=[str(d) for d in dropout_rates])
axes[0].set_xlabel('Dropout Rate', fontsize=12)
axes[0].set_ylabel('Test R²', fontsize=12)
axes[0].set_title('MLP: Dropout vs Performance', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# GNN
gnn_dropout_groups = [gnn_data[gnn_data['Dropout'] == d]['R² (Test)'].values 
                      for d in dropout_rates]
axes[1].boxplot(gnn_dropout_groups, labels=[str(d) for d in dropout_rates])
axes[1].set_xlabel('Dropout Rate', fontsize=12)
axes[1].set_ylabel('Test R²', fontsize=12)
axes[1].set_title('GNN: Dropout vs Performance', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Random vs Scaffold split comparison
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

x = np.arange(4)
width = 0.35

best_random_vals = []
best_scaffold_vals = []
labels = []

for model in ['MLP', 'GNN']:
    random_best = results_df[(results_df['Model'] == model) & 
                            (results_df['Split Type'] == 'Random')]['R² (Test)'].max()
    scaffold_best = results_df[(results_df['Model'] == model) & 
                              (results_df['Split Type'] == 'Scaffold')]['R² (Test)'].max()
    best_random_vals.append(random_best)
    best_scaffold_vals.append(scaffold_best)
    labels.append(model)

ax.bar(x[:len(labels)] - width/2, best_random_vals, width, label='Random Split', alpha=0.8)
ax.bar(x[:len(labels)] + width/2, best_scaffold_vals, width, label='Scaffold Split', alpha=0.8)

ax.set_xlabel('Model Type', fontsize=12)
ax.set_ylabel('Best Test R²', fontsize=12)
ax.set_title('Best Performance: Random vs Scaffold Split', fontsize=14, fontweight='bold')
ax.set_xticks(x[:len(labels)])
ax.set_xticklabels(labels)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion

We tested 36 different configurations across MLP and GNN models on both random and scaffold splits.

**Key findings:**
1. Hyperparameter tuning matters! Different configs show significant performance variation.
2. Scaffold split is harder than random split (as expected) - testing true generalization.
3. Best configurations identified for each model/split combination.
4. Learning rate and dropout both significantly impact performance.

**Next steps:**
- Use the best configurations for production models
- Consider more advanced architectures (GAT, MPNN) for GNN
- Try additional features or data augmentation
- Implement ensemble models combining best configs